# Act 2 · Hybrid Search (`AI.SEARCH` + RRF) & 2-Hop SQL Graph Features

### Why You Don't Need a PyTorch GraphSAGE Pipeline for Entity Resolution
A Graph Neural Network (like a 2-layer GraphSAGE link predictor) aggregates features from a node's **1-hop** and **2-hop** neighbourhood in the bipartite `Profile <-> Identifier` graph. Why?
1. **1-hop (`Profile -> Shared Identifier`):** How rare is the identifier connecting Record A and Record B?
2. **2-hop (`Profile A -> Other Identifiers of A` vs `Profile B -> Other Identifiers of B`):** Do Record A and Record B also carry *different*, non-overlapping emails, phones, or loyalty accounts (`unshared_email_both`, `unshared_phone_both`, `unshared_acct_both`), or conflicting street addresses (`street_conflict`)?

Because our bipartite graph has a known schema (`Profile <-> Identifier`), we compute **the exact 1-hop and 2-hop sufficient statistics deterministically in BigQuery SQL (`40_block.sql` + `50_candidates.sql`)**, and combine them with **BigQuery Hybrid Search (`AI.SEARCH` + `VECTOR_SEARCH` + Reciprocal Rank Fusion)**:
- **Zero GPU training pipelines or stale node embeddings** when new profiles arrive.
- **100% explainable features** (`idf_weight_sum`, `unshared_email_both`, `forename_conflict`) passed directly into the rule scorer and Gemini 3.5 Flash prompt.
- **Higher recall on disconnected graph components:** A GNN link predictor can only score pairs that already share a path in the identifier graph. BigQuery `VECTOR_SEARCH` retrieves transliterated names, typos, and unstructured call transcripts even when **zero exact identifiers match (`SEMANTIC_ONLY`)**.

In [1]:
%load_ext bigquery_magics

import bigquery_magics
import pandas as pd
from IPython.display import display
from pathlib import Path

# ---------------------------------------------------------------------------
# Set PROJECT if running outside a checkout that has demo/config.env
PROJECT = ""          # e.g. "all-things-cdp"
DATASET = ""          # leave blank for the default, "cdp"
# ---------------------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 52)

CFG, CONFIG, _source = {}, None, None

def _find_config():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        for candidate in (base / "config.env", base / "demo" / "config.env"):
            if candidate.is_file():
                return candidate
    return None

if not PROJECT:
    CONFIG = _find_config()
    if CONFIG:
        for _line in CONFIG.read_text().splitlines():
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                CFG[_k.strip()] = _v.split("#", 1)[0].strip().strip('"').strip("'")
        PROJECT = CFG.get("CDP_PROJECT", "")
        _source = str(CONFIG)

if not PROJECT:
    import os
    PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT") or os.environ.get("GCP_PROJECT") or "all-things-cdp"
    _source = "environment / default"

DATASET = DATASET or CFG.get("CDP_DS") or CFG.get("CDP_DATASET_PREFIX") or "cdp"
LOCATION = CFG.get("CDP_LOCATION", "US")

bigquery_magics.context.project = PROJECT
bigquery_magics.context.location = LOCATION
bigquery_magics.context.progress_bar_type = None

print(f"project    {PROJECT}   (from {_source})")
print(f"dataset    {DATASET}   (location {LOCATION})")

project    all-things-cdp   (from demo/config.env)
dataset    cdp   (location US)

## 1 · Two-Leg Candidate Retrieval: Lexical Blocking vs Semantic Vector Search (`cdp.v_retrieval_legs`)

Stage 40 (`40_block.sql`) retrieves candidate pairs via two independent legs and fuses their rankings in Stage 50 (`50_candidates.sql`) using **Reciprocal Rank Fusion (`1/(60 + rank_lexical) + 1/(60 + rank_semantic)`)**:
- **`LEXICAL_ONLY`:** Exact match on normalized email, E.164 phone, loyalty account, postcode + surname Soundex, or DOB + surname prefix.
- **`SEMANTIC_ONLY`:** Top-$K$ nearest neighbours in `VECTOR_SEARCH(TABLE cdp.party_search, 'match_embedding', ...)` with **no shared lexical blocking key**.
- **`BOTH`:** Found by both the lexical blocking leg and the vector search leg.

In [2]:
%%bigquery df_legs
SELECT
  retrieved_by,
  pairs_retrieved,
  became_auto_match,
  became_grey_zone,
  became_reject,
  avg_score,
  avg_rrf,
  with_shared_account,
  surname_differs_but_kept
FROM `cdp.v_retrieval_legs`
ORDER BY pairs_retrieved DESC

retrieved_by,pairs_retrieved,became_auto_match,became_grey_zone,became_reject,avg_score,avg_rrf,with_shared_account,surname_differs_but_kept
LEXICAL_ONLY,10445365,6,31003,10414356,0.025,0.0154,27,28545
SEMANTIC_ONLY,244079,25,1982,242072,0.311,0.014,0,1254
BOTH,84398,4314,8784,71300,0.339,0.03,1213,5234


## 2 · Inspecting the 2-Hop Neighbourhood Features in `cdp.pair_features`

Let's query `cdp.pair_features` to see how the **1-hop rarity features** (`idf_weight_sum`, `min_shared_degree`, `via_hub`, `hub_only`) and **2-hop neighbourhood features** (`unshared_email_both`, `unshared_phone_both`, `unshared_acct_both`, `street_match`, `street_conflict`) separate true matches from household collisions and supersession traps.

In [3]:
%%bigquery df_2hop
SELECT
  CASE
    WHEN hub_only THEN '1. Promiscuous Hub Collision (hub_only)'
    WHEN (email_match OR phone_match) AND forename_conflict THEN '2. Household Shared Email/Phone (forename_conflict)'
    WHEN (email_match OR phone_match) AND (unshared_email_both OR unshared_phone_both) THEN '3. 2-Hop Divergent Secondary IDs (unshared_*_both)'
    WHEN email_match AND phone_match AND NOT forename_conflict THEN '4. Multi-Signal True Match (email + phone)'
    ELSE '5. Other Candidate Pairs'
  END AS neighbourhood_pattern,
  COUNT(*) AS candidate_pairs,
  ROUND(AVG(idf_weight_sum), 2) AS avg_idf_weight_sum,
  ROUND(AVG(rule_score), 3) AS avg_rule_score,
  ROUND(AVG(combined_score), 3) AS avg_combined_score
FROM `cdp.pair_features`
GROUP BY neighbourhood_pattern
ORDER BY neighbourhood_pattern

neighbourhood_pattern,candidate_pairs,avg_idf_weight_sum,avg_rule_score,avg_combined_score
1. Promiscuous Hub Collision (hub_only),2025,0.0,0.0,0.007
2. Household Shared Email/Phone (forename_conflict),445,9.44,0.653,0.708
3. 2-Hop Divergent Secondary IDs (unshared_*_both),124,9.08,0.839,0.868
4. Multi-Signal True Match (email + phone),557,18.21,0.997,0.997
5. Other Candidate Pairs,10770691,0.0,0.027,0.033


## 3 · The Candidate Funnel (`cdp.v_candidate_funnel`)

By combining the 1-hop/2-hop graph features with RRF hybrid search scores, Stage 50 (`50_candidates.sql`) routes the **10,773,842** candidate pairs into three tiers:
- **`AUTO_MATCH` (`4,345` pairs, `0.04%`):** High-rarity corroborating identifiers with zero 2-hop conflicts — merged immediately with **zero LLM cost**.
- **`REJECT` (`10,727,728` pairs, `99.57%`):** Filtered out deterministically with **zero LLM cost**.
- **`GREY_ZONE` (`41,769` pairs, `0.39%`):** Ambiguous pairs where evidence points both ways — routed to **Gemini 3.5 Flash** in Stage 60.

In [4]:
%%bigquery df_funnel
SELECT
  tier,
  pairs,
  pct,
  min_score,
  avg_score,
  max_score,
  semantic_only,
  lexical_only,
  both
FROM `cdp.v_candidate_funnel`
ORDER BY CASE tier WHEN 'AUTO_MATCH' THEN 1 WHEN 'GREY_ZONE' THEN 2 ELSE 3 END

tier,pairs,pct,min_score,avg_score,max_score,semantic_only,lexical_only,both
AUTO_MATCH,4345,0.04,0.409,0.926,1.0,25,6,4314
GREY_ZONE,41769,0.39,0.15,0.267,1.0,1982,31003,8784
REJECT,10727728,99.57,0.0,0.032,1.0,242072,10414356,71300
